# 02 — Run MiDaS on KITTI-C

Batch inference using `dpt_hybrid_384` over the KITTI-C manifest.
Predictions are saved as `.npy` files in `outputs/predictions/kitti_c/`.

**Requires:** manifest from notebook 01 and KITTI-C data in place.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
# --- Config ---
MODEL_TYPE = 'dpt_hybrid_384'
MANIFEST_PATH = PROJECT_ROOT / 'data' / 'manifests' / 'kitti_c_manifest.csv'
PRED_DIR = PROJECT_ROOT / 'outputs' / 'predictions' / 'kitti_c'

# Set to a small number to test; set to None for full run
MAX_SAMPLES = 50

In [ ]:
import pandas as pd
from src.adapters.midas_adapter import MiDaSAdapter

df = pd.read_csv(MANIFEST_PATH)
df = df[df['gt_path'].notna()].reset_index(drop=True)

if MAX_SAMPLES:
    df = df.sample(n=min(MAX_SAMPLES, len(df)), random_state=42).reset_index(drop=True)

print(f'Running inference on {len(df)} images')

adapter = MiDaSAdapter(model_type=MODEL_TYPE)

In [ ]:
import numpy as np
from tqdm.notebook import tqdm

skipped = 0
for _, row in tqdm(df.iterrows(), total=len(df)):
    out_path = PRED_DIR / row['corruption_type'] / str(row['severity']) / f"{row['frame_id']}.npy"
    if out_path.exists():
        skipped += 1
        continue
    out_path.parent.mkdir(parents=True, exist_ok=True)
    depth = adapter.predict(row['image_path'])
    np.save(str(out_path), depth)

print(f'Done. Skipped {skipped} already-existing predictions.')

## Next step

Run `03_eval_metrics.ipynb` to compute alignment and depth metrics.